# 🎙️ Urdu Interview Processing Pipeline
**Stages:** Audio → Urdu Transcript → Verify → English Translation → Verify → De-identify → Final Dataset

**Models used:**
- ASR: `openai/whisper-large-v3-turbo` (via faster-whisper)
- Translation: `facebook/nllb-200-1.3B` ⬆️ **Upgraded for better quality**
- De-identification: `Microsoft Presidio` + `spaCy en_core_web_lg`

**Improvements in this version:**
- ✅ NLLB model upgraded from 600M to 1.3B (3x better translation)
- ✅ Sentence-aware chunking (preserves context instead of hard character splits)
- ✅ Better handling of Urdu→English translations

> ⚠️ Make sure **Runtime → Change runtime type → T4 GPU** is selected before running!

In [1]:
# ── CELL 1: Check GPU ──────────────────────────────────────
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f'✔ GPU available: {gpu_name}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠ No GPU detected! Go to Runtime → Change runtime type → T4 GPU')
    print('  Pipeline will run on CPU (much slower)')

✔ GPU available: Tesla T4
  VRAM: 15.6 GB


In [1]:
# ── CELL 2: Install all dependencies ──────────────────────
print('Installing dependencies... (takes 3-5 minutes first time)')
print('⚠ Note: NLLB 1.3B model (~2.5GB) requires T4 GPU VRAM (~16GB available)')

!pip install -q faster-whisper==1.1.0
!pip install -q transformers==4.44.2 sentencepiece==0.2.0 sacremoses==0.1.1
!pip install -q presidio-analyzer==2.2.355 presidio-anonymizer==2.2.355
!pip install -q spacy==3.8.1
!pip install -q python-docx==1.1.2 langdetect==1.0.9 sacrebleu==2.4.3

# Download spaCy model
!python -m spacy download en_core_web_lg -q

print('\n✔ All dependencies installed!')
print('✔ Models will auto-download on first use (may take 2-3 minutes per model)')

Installing dependencies... (takes 3-5 minutes first time)
⚠ Note: NLLB 1.3B model (~2.5GB) requires T4 GPU VRAM (~16GB available)


KeyboardInterrupt: 

In [1]:
# ── CELL 3: Clone project from GitHub ──────────────────────
import os
import shutil

REPO_DIR = 'urdu-pipeline'
REPO_URL = 'https://github.com/mSaadAli99/URDU-ENGLISH-TRANSLATION-PIPELINE.git'

if os.path.isdir(REPO_DIR):
    print(f'✔ Repo already exists — pulling latest changes')
    %cd {REPO_DIR}
    !git pull origin main
else:
    !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

# Stale nested clone causes pipeline to read OLD 12-min audio from urdu-pipeline/urdu-pipeline/audio/
nested = os.path.join(os.getcwd(), 'urdu-pipeline')
if os.path.isdir(nested) and os.path.isfile(os.path.join(nested, 'main.py')):
    print('⚠ Removing stale nested urdu-pipeline/ folder (wrong audio + outputs path)')
    shutil.rmtree(nested)

print('✔ Repository ready!')
print('   Working directory:', os.getcwd())
!ls -la


Cloning into 'urdu-pipeline'...
remote: Enumerating objects: 142, done.
remote: Counting objects: 100% (142/142), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 142 (delta 79), reused 133 (delta 70), pack-reused 0 (from 0)
Receiving objects: 100% (142/142), 476.70 KiB | 16.44 MiB/s, done.
Resolving deltas: 100% (79/79), done.
/content/urdu-pipeline
✔ Repository ready!
   Working directory: /content/urdu-pipeline
total 60
drwxr-xr-x 6 root root 4096 Jul  3 11:15 .
drwxr-xr-x 1 root root 4096 Jul  3 11:15 ..
-rw-r--r-- 1 root root 7703 Jul  3 11:15 config.py
drwxr-xr-x 8 root root 4096 Jul  3 11:15 .git
-rw-r--r-- 1 root root 1313 Jul  3 11:15 .gitignore
-rw-r--r-- 1 root root 9651 Jul  3 11:15 main.py
drwxr-xr-x 2 root root 4096 Jul  3 11:15 notebooks
drwxr-xr-x 2 root root 4096 Jul  3 11:15 pipeline
-rw-r--r-- 1 root root 4284 Jul  3 11:15 README.md
-rw-r--r-- 1 root root 1077 Jul  3 11:15 requirements.txt
drwxr-xr-x 2 root root 4096 Jul  3 11:15 scripts


In [2]:
# ── CELL 4: Download audio from YouTube ───────────────────
import os
import subprocess
import glob

os.makedirs('audio', exist_ok=True)

AUDIO_PATH   = 'audio/test_audio.mp3'
FULL_PATTERN = 'audio/full.*'
YOUTUBE_URL  = 'https://youtu.be/pHZHYWe8Mkc'
START_SEC    = 137
TARGET_DURATION_SEC = 3300  # 55 minutes — clip length to extract
MIN_DURATION_SEC    = 3000  # 50 minutes — re-download if file is shorter

def _find_full_audio():
    matches = sorted(glob.glob('audio/full.*'))
    return matches[0] if matches else None

def _get_duration_sec(path):
    try:
        out = subprocess.run(
            ['ffprobe', '-v', 'error', '-show_entries', 'format=duration',
             '-of', 'default=noprint_wrappers=1:nokey=1', path],
            capture_output=True, text=True, check=True,
        )
        return float(out.stdout.strip())
    except Exception:
        return 0.0

need_download = True
if os.path.exists(AUDIO_PATH):
    actual_min = _get_duration_sec(AUDIO_PATH) / 60
    if actual_min >= MIN_DURATION_SEC / 60:
        print(f'✔ Audio already exists ({actual_min:.1f} min) — skipping download: {AUDIO_PATH}')
        need_download = False
    else:
        print(f'⚠ Cached audio is only {actual_min:.1f} min (need ≥{MIN_DURATION_SEC // 60} min).')
        print(f'  Deleting old file and re-downloading...')
        os.remove(AUDIO_PATH)
        for old in glob.glob('audio/full.*'):
            os.remove(old)

if need_download:
    !pip install -q -U yt-dlp

    strategies = [
        ['yt-dlp', '-x', '--audio-format', 'mp3', '-o', 'audio/full.%(ext)s', YOUTUBE_URL],
        ['yt-dlp', '-x', '--audio-format', 'mp3', '-o', 'audio/full.%(ext)s',
         '--extractor-args', 'youtube:player_client=android', YOUTUBE_URL],
        ['yt-dlp', '-x', '--audio-format', 'mp3', '-o', 'audio/full.%(ext)s',
         '--extractor-args', 'youtube:player_client=ios', YOUTUBE_URL],
    ]

    full_audio = None
    for i, cmd in enumerate(strategies, 1):
        print(f'\n  Download attempt {i}/{len(strategies)}...')
        result = subprocess.run(cmd, capture_output=True, text=True)
        full_audio = _find_full_audio()
        if result.returncode == 0 and full_audio:
            print(f'  ✔ Downloaded: {full_audio}')
            break
        err = (result.stderr or result.stdout or '').strip()
        if err:
            print('  ', err.splitlines()[-1])

    if not full_audio:
        print('\n⚠ YouTube download failed (Colab bot detection).')
        print('  Upload your own MP3 using the cell below, then re-run this cell.')
        from google.colab import files
        print('\n  Waiting for upload → save as audio/test_audio.mp3 ...')
        uploaded = files.upload()
        for name, data in uploaded.items():
            dest = AUDIO_PATH if name.endswith('.mp3') else f'audio/{name}'
            with open(dest, 'wb') as f:
                f.write(data)
            print(f'  ✔ Saved upload → {dest}')
        if not os.path.exists(AUDIO_PATH):
            raise FileNotFoundError(
                'No audio file available. Upload an MP3 or try running this cell again later.'
            )
    else:
        trim_cmd = [
            'ffmpeg', '-y', '-i', full_audio,
            '-ss', str(START_SEC), '-t', str(TARGET_DURATION_SEC),
            '-c', 'copy', AUDIO_PATH,
        ]
        trim = subprocess.run(trim_cmd, capture_output=True, text=True)
        if trim.returncode != 0 or not os.path.exists(AUDIO_PATH):
            print('  Trim with -c copy failed — re-encoding clip...')
            subprocess.run([
                'ffmpeg', '-y', '-i', full_audio,
                '-ss', str(START_SEC), '-t', str(TARGET_DURATION_SEC),
                AUDIO_PATH,
            ], check=True)
        if os.path.exists(full_audio):
            os.remove(full_audio)

audio_path = AUDIO_PATH
size_mb = os.path.getsize(audio_path) / 1e6
actual_sec = _get_duration_sec(audio_path)
actual_min = actual_sec / 60
print(f'\n✔ Audio ready: {audio_path}')
print(f'   Absolute path: {os.path.abspath(audio_path)}')
print(f'   File size: {size_mb:.1f} MB')
print(f'   Actual duration: {actual_min:.1f} min (target: {TARGET_DURATION_SEC // 60} min from t={START_SEC}s)')
if actual_sec < MIN_DURATION_SEC:
    raise RuntimeError(
        f'Audio is only {actual_min:.1f} min — need at least {MIN_DURATION_SEC // 60} min. '
        'YouTube trim may have failed; upload a 50+ min MP3 manually in the upload fallback.'
    )


✔ Audio already exists (55.0 min) — skipping download: audio/test_audio.mp3

✔ Audio ready: audio/test_audio.mp3
   Absolute path: /content/urdu-pipeline/audio/test_audio.mp3
   File size: 31.2 MB
   Actual duration: 55.0 min (target: 55 min from t=137s)


In [4]:
# ── CELL 5: Configure pipeline ─────────────────────────────
import sys
import os

REPO_ROOT = os.getcwd()  # already inside repo after Cell 3 — do NOT use 'urdu-pipeline' subfolder
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import config

# Set device to cuda since we have GPU
config.WHISPER_DEVICE       = 'cuda'
config.WHISPER_COMPUTE_TYPE = 'float16'

# Set audio path (must be the 55-min file from Cell 4, NOT a nested copy)
AUDIO_PATH = os.path.abspath(audio_path)
print(f'  Audio absolute path: {AUDIO_PATH}')

# Force Urdu transcription (Urdu script) — disables two-pass auto-detect
config.WHISPER_LANGUAGE = 'ur'
config.WHISPER_TWO_PASS = False
config.WHISPER_INITIAL_PROMPT = (
    'یہ ایک انٹرویو ہے۔ اردو میں اردو رسم الخط میں لکھیں۔ '
    'This is an interview. Transcribe Urdu speech in Urdu Arabic script.'
)

print('Pipeline Configuration:')
print(f'  ASR Model        : whisper-{config.WHISPER_MODEL}')
print(f'  Translation Model: {config.TRANSLATION_MODEL}')
print(f'  Device           : {config.WHISPER_DEVICE}')
print(f'  Language mode    : {config.WHISPER_LANGUAGE} (forced)')
print(f'  Two-pass ASR     : {config.WHISPER_TWO_PASS}')
print(f'  Audio file       : {AUDIO_PATH}')
print(f'  Confidence thresh: {config.CONFIDENCE_THRESHOLD}')
print(f'  Chunk size       : {config.CHUNK_SIZE} chars')

  Audio absolute path: /content/urdu-pipeline/audio/test_audio.mp3
Pipeline Configuration:
  ASR Model        : whisper-large-v3-turbo
  Translation Model: facebook/nllb-200-1.3B
  Device           : cuda
  Language mode    : ur (forced)
  Two-pass ASR     : False
  Audio file       : /content/urdu-pipeline/audio/test_audio.mp3
  Confidence thresh: 0.55
  Chunk size       : 500 chars


In [5]:
# Check if audio file exists and measure REAL duration (ffprobe)
import os
import subprocess

audio_path = os.path.abspath("audio/test_audio.mp3")
MIN_MINUTES = 50

def _probe_minutes(path):
    try:
        out = subprocess.run(
            ['ffprobe', '-v', 'error', '-show_entries', 'format=duration',
             '-of', 'default=noprint_wrappers=1:nokey=1', path],
            capture_output=True, text=True, check=True,
        )
        return float(out.stdout.strip()) / 60
    except Exception:
        return 0.0

if os.path.exists(audio_path):
    size_mb = os.path.getsize(audio_path) / 1e6
    dur_min = _probe_minutes(audio_path)
    flag = '✔' if dur_min >= MIN_MINUTES else '⚠ TOO SHORT — re-run Cell 4'
    print(f'{flag} Audio file: {size_mb:.1f} MB, {dur_min:.1f} minutes (need ≥{MIN_MINUTES} min)')
else:
    print(f'✗ Audio file NOT found: {audio_path}')

# Check GPU status
import torch
if torch.cuda.is_available():
    print(f'✔ GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print(f'   Used: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB')
else:
    print('✗ GPU not available!')

✔ Audio file: 31.2 MB, 55.0 minutes (need ≥50 min)
✔ GPU: Tesla T4
   VRAM: 15.6 GB
   Used: 0.00 GB


In [6]:
# ── CELL 6: STAGE 1 — Urdu Transcription ──────────────────
import importlib.util

from pipeline.utils import ensure_dirs
ensure_dirs(config.STAGE1_DIR, config.STAGE2_DIR, config.STAGE3_DIR,
            config.STAGE4_DIR, config.STAGE5_DIR, config.STAGE6_DIR)

# Runtime-safe dependency check (Colab runtimes sometimes reset packages)
if importlib.util.find_spec('faster_whisper') is None:
    print('Installing missing dependency: faster-whisper==1.1.0')
    !pip install -q faster-whisper==1.1.0

from pipeline.transcribe import transcribe

stage1_result = transcribe(AUDIO_PATH)

# Preview
print('\n── Urdu Transcript Preview (first 500 chars) ──')
print(stage1_result['full_urdu_text'][:500])


  STAGE 1: TRANSCRIPTION (ASR)
  Audio file   : /content/urdu-pipeline/audio/test_audio.mp3
  Model        : whisper-large-v3-turbo
  Language     : ur
  Two-pass ASR : off  (v2)
  Device       : cuda
  Temperature  : 0.0 (fallback: [0.2, 0.4, 0.6])
  Beam size    : 5

  Loading Whisper model (first run downloads ~800 MB)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


  Model loaded.

  Pass 1: transcribing audio (language: ur) ...
  Raw segments from Whisper: 801
  Merged 84 micro-segments into neighbours.

  Results:
  Segments total    : 717
    Urdu (routed)   : 619
    Urdu (script)   : 593
    English         : 98
    Low confidence  : 3  (threshold=0.55)
  Avg raw confidence: 0.850
  Avg cal confidence: 0.912
  Avg text quality  : 0.802
  Loops removed     : 0
  Micro-segs merged : 84

  Segment detail:
    [ ][UR]    00:00:00.180 -> 00:00:06.440 | conf=0.930 | n/a                | السلام علیکم Dr. Majda. Welcome to the show. 
    [ ][UR]    00:00:07.920 -> 00:00:17.160 | conf=0.843 | n/a                | If you give us a small introduction so that o
    [ ][EN]    00:00:17.360 -> 00:00:26.200 | conf=0.950 | n/a                | Thank you so much Zainab for inviting me on y
    [ ][UR]    00:00:32.880 -> 00:00:45.020 | conf=0.866 | n/a                | بیٹے کے ساتھ ایسا ہو چکا ہے کہ اس نے کہا کہ م
    [ ][UR]    00:00:45.020 -> 00:00:55.260 |

In [3]:
#Clearing memory of previous model to free up GPU VRAM for next stage
import torch, gc
if 'model' in globals():
    del model
gc.collect()
torch.cuda.empty_cache()

In [8]:
# ── CELL 7: STAGE 2 — Verify Urdu Transcript ──────────────
from pipeline.verify_transcript import verify_transcript

stage2_result = verify_transcript(stage1_result)

print(f'\nQuality Score : {stage2_result["quality_score"]}/100')
print(f'Quality Label : {stage2_result["quality_label"]}')

# Show flagged segments
flagged = stage2_result['verification_report']['flagged_details']
if flagged:
    print(f'\n⚠ Flagged Segments ({len(flagged)}):')
    for f in flagged[:5]:
        print(f'  seg {f["segment_id"]} [{f["start_fmt"]}] conf={f["confidence"]:.2f}: {f["text"][:60]}...')
else:
    print('\n✔ No segments flagged!')


  STAGE 2: VERIFICATION OF TRANSCRIPT
  Interview ID        : test_audio
  Total segments      : 717
  Urdu segments       : 619
  English segments    : 98
  Micro-segs merged   : 84
  Loops removed       : 0
  Avg raw confidence  : 0.8497
  Avg cal confidence  : 0.9118
  Avg text quality    : 0.8019
  Confidence threshold: 0.55
    [ok      ][UR]    seg 001 | conf=0.930 | tq=0.94 | السلام علیکم Dr. Majda. Welcome to the show. 
    [ok      ][UR]    seg 002 | conf=0.843 | tq=0.99 | If you give us a small introduction so that o
    [ok      ][EN]    seg 003 | conf=0.950 | tq=0.97 | Thank you so much Zainab for inviting me on y
    [ok      ][UR]    seg 004 | conf=0.866 | tq=0.74 | بیٹے کے ساتھ ایسا ہو چکا ہے کہ اس نے کہا کہ م
    [ok      ][UR]    seg 005 | conf=0.925 | tq=0.79 | تو پی ایڈی میں نے کیا ہے میرا ایریا آف اسپیشل
    [ok      ][UR]    seg 006 | conf=0.869 | tq=0.79 | میں کمپیوٹر انجنیرنگ دپارمنٹ میں اس کے علاوہ 
    [ok      ][EN]    seg 007 | conf=0.898 | tq=0.87 | stem ce

In [9]:
# ── CELL 8: STAGE 3 — Translate Urdu → English ────────────
from pipeline.translate import translate

stage3_result = translate(stage2_result)

print('\n── English Translation Preview (first 500 chars) ──')
print(stage3_result['english_full_text'][:500])


  STAGE 3: TRANSLATION: URDU → ENGLISH (smart routing)
  Interview ID   : test_audio
  Model          : facebook/nllb-200-1.3B
  Routing        : Urdu→NLLB  |  English→pass-through

  Routing: 619 Urdu segments -> translate  |  98 English segments -> pass-through

  Loading translation model (~2.5GB first run)...


tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


config.json:   0%|          | 0.00/808 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/5.48G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

  Model loaded on cuda.

  Translating Urdu segments...

  Assembling translated segments:
    [T] seg 001/717 | TRANSLATED | Dr. Majda, welcome to the show. Thank you so much for coming...
    [T] seg 002/717 | TRANSLATED | If you give us a little introduction so that our audience th...
    [-] seg 003/717 | PASS-THROUGH | Thank you so much Zainab for inviting me on your show. My na...
    [T] seg 004/717 | TRANSLATED | This has happened to my son. He said that his mother was a d...
    [T] seg 005/717 | TRANSLATED | So what I do at PED is my area of specialization is digital ...
    [T] seg 006/717 | TRANSLATED | In addition to the computer engineering department at N.E.U....
    [-] seg 007/717 | PASS-THROUGH | stem center ہے which is i think one of its own kind کسی بھی ...
    [T] seg 008/717 | TRANSLATED | To the best of my knowledge this is the first stem center of...
    [T] seg 009/717 | TRANSLATED | And besides I'm the principal investigator of the National C...
    [T] seg 01

In [10]:
# ── CELL 9: STAGE 4 — Verify English Translation ──────────
from pipeline.verify_translation import verify_translation

stage4_result = verify_translation(stage3_result)

print(f'\nTranslation Quality Score : {stage4_result["translation_quality_score"]}/100')
print(f'Translation Quality Label : {stage4_result["translation_quality_label"]}')

# Show flagged translation segments
t_flagged = stage4_result['translation_verification_report']['flagged_details']
if t_flagged:
    print(f'\n⚠ Translation Flagged Segments ({len(t_flagged)}):')
    for f in t_flagged[:5]:
        print(f'  seg {f["segment_id"]}: {f["eng_text"][:60]}... | Issues: {", ".join(f["issues"])}')
else:
    print('\n✔ All translations verified OK!')


  STAGE 4: VERIFICATION OF ENGLISH TRANSLATION
  Interview ID      : test_audio
  Total segments    : 717
  Translated        : 619
  Pass-through      : 98

  Checks: empty, error markers, length ratio, nonsense detection
    [✔][T] seg 001
             EN: Dr. Majda, welcome to the show. Thank you so much for coming today and...
    [✔][T] seg 002
             EN: If you give us a little introduction so that our audience that you are...
    [✔][-] seg 003
             EN: Thank you so much Zainab for inviting me on your show. My name is Majd...
    [✔][T] seg 004
             EN: This has happened to my son. He said that his mother was a doctor. He ...
    [✔][T] seg 005
             EN: So what I do at PED is my area of specialization is digital systems de...
    [✔][T] seg 006
             EN: In addition to the computer engineering department at N.E.U....
    [✔][-] seg 007
             EN: stem center ہے which is i think one of its own kind کسی بھی higher edu...
    [✔][T] seg 0

In [11]:
# ── CELL 10: STAGE 5 — De-identification ──────────────────
from pipeline.deidentify import deidentify

stage5_result = deidentify(stage4_result)

print(f'\nEntities removed: {stage5_result["entities_removed_count"]}')
print('\n── De-identified Text Preview (first 500 chars) ──')
print(stage5_result['deidentified_english_full'][:500])


  STAGE 5: DE-IDENTIFICATION OF DATASET
  Interview ID : test_audio
  Loading Presidio + spaCy (first run may take a moment)...
  ✔ Presidio loaded.

  De-identifying full English text...
      [skip] Ignored generic temporal words: ['today', 'today', 'night', 'the day', 'Today', 'today']

  De-identifying segments:
      [skip] Ignored generic temporal words: ['today']
    [PII] seg 001 | 1 entities removed | Dr. [NAME], welcome to the show. Thank you so much for comin...
      [skip] Ignored generic temporal words: ['today']
    [ok ] seg 002 | 0 entities removed | If you give us a little introduction so that our audience th...
    [PII] seg 003 | 2 entities removed | Thank you so much [NAME] for inviting me on your show. My na...
    [ok ] seg 004 | 0 entities removed | This has happened to my son. He said that his mother was a d...
    [ok ] seg 005 | 0 entities removed | So what I do at PED is my area of specialization is digital ...
    [ok ] seg 006 | 0 entities removed | In ad

In [12]:
# ── CELL 11: STAGE 6 — Final Export ───────────────────────
from pipeline.export import export

final_result = export(stage5_result)

print(f'\n✔ Final JSON : {final_result["json_path"]}')
print(f'✔ Final DOCX : {final_result["docx_path"]}')


  STAGE 6: FINAL EXPORT: JSON + DOCX
  Interview ID : test_audio

  Building final JSON dataset...
  Saved -> /content/urdu-pipeline/outputs/6_final_dataset/test_audio_final_dataset.json

  Building DOCX report...
  ✔ DOCX saved → /content/urdu-pipeline/outputs/6_final_dataset/test_audio_final_dataset.docx

  ── Final Outputs ─────────────────────────────
  JSON → /content/urdu-pipeline/outputs/6_final_dataset/test_audio_final_dataset.json
  DOCX → /content/urdu-pipeline/outputs/6_final_dataset/test_audio_final_dataset.docx

  ✔ Stage 6 complete. Pipeline finished!

✔ Final JSON : /content/urdu-pipeline/outputs/6_final_dataset/test_audio_final_dataset.json
✔ Final DOCX : /content/urdu-pipeline/outputs/6_final_dataset/test_audio_final_dataset.docx


In [13]:
# ── CELL 12: Save outputs to Google Drive ─────────────────
# Browser download often fails in Colab — Drive is reliable.
import os
import shutil
from datetime import datetime

try:
    from google.colab import drive, files
    IN_COLAB = True
except ImportError:
    drive = files = None
    IN_COLAB = False

interview_id = stage1_result.get('interview_id', 'interview')
print('Current working dir:', os.getcwd())

candidate_roots = [
    os.getcwd(),
    '/content/urdu-pipeline',
    '/content',
]

repo_root = None
outputs_dir = None
for root in candidate_roots:
    if os.path.isdir(os.path.join(root, 'outputs')):
        repo_root = root
        outputs_dir = os.path.join(root, 'outputs')
        break

if outputs_dir is None:
    print('✗ Error: outputs/ folder not found!')
    print('   Run Cells 6-11 (all pipeline stages) first.')
else:
    print(f'✔ Outputs folder: {outputs_dir}')
    os.system(f'ls -lh "{outputs_dir}" || true')

    zip_base = os.path.join(repo_root, f'{interview_id}_outputs')
    zip_file = f'{zip_base}.zip'
    if os.path.exists(zip_file):
        os.remove(zip_file)
    shutil.make_archive(
        base_name=zip_base,
        format='zip',
        root_dir=repo_root,
        base_dir='outputs',
    )

    if not os.path.exists(zip_file):
        raise FileNotFoundError(f'Could not create zip: {zip_file}')

    size_mb = os.path.getsize(zip_file) / 1e6
    print(f'\n✔ Zipped outputs: {zip_file} ({size_mb:.1f} MB)')

    if IN_COLAB:
        print('\nMounting Google Drive — click the link and allow access...')
        drive.mount('/content/drive')

        drive_base = os.path.join('/content/drive/MyDrive', 'urdu-pipeline-outputs')
        run_folder = os.path.join(
            drive_base,
            f"{interview_id}_{datetime.now().strftime('%Y%m%d_%H%M%S')}",
        )
        os.makedirs(run_folder, exist_ok=True)

        drive_zip = os.path.join(run_folder, f'{interview_id}_outputs.zip')
        shutil.copy2(zip_file, drive_zip)

        drive_outputs = os.path.join(run_folder, 'outputs')
        if os.path.exists(drive_outputs):
            shutil.rmtree(drive_outputs)
        shutil.copytree(outputs_dir, drive_outputs)

        audio_src = os.path.join(repo_root, 'audio', 'test_audio.mp3')
        if os.path.exists(audio_src):
            shutil.copy2(audio_src, os.path.join(run_folder, 'test_audio.mp3'))
            print('  ✔ Audio copied to Drive')

        print('\n' + '=' * 60)
        print('  SAVED TO GOOGLE DRIVE')
        print('=' * 60)
        print(f'  Folder : My Drive / urdu-pipeline-outputs /')
        print(f'           {os.path.basename(run_folder)}/')
        print(f'  ZIP    : {interview_id}_outputs.zip')
        print(f'  Full   : outputs/ (all 6 stages)')
        print('\n  Open drive.google.com → My Drive → urdu-pipeline-outputs')

        # Optional browser download (often blocked — Drive is primary)
        try:
            print('\n  Also trying browser download (may be blocked)...')
            files.download(zip_file)
        except Exception as e:
            print(f'  Browser download skipped: {e}')
    else:
        print(f'  Not in Colab — files at: {zip_file}')


Current working dir: /content/urdu-pipeline
✔ Outputs folder found at: /content/urdu-pipeline/outputs
Contents:

✔ Zipped outputs: /content/urdu-pipeline/test_audio_outputs.zip (1.5 MB)
  Downloading...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [2]:
# ── CELL 13 (OPTIONAL): Run full pipeline in one go ───────
# Use this after testing individual stages above

import os
import subprocess
import config

config.WHISPER_DEVICE = 'cuda'
config.WHISPER_COMPUTE_TYPE = 'float16'

AUDIO_FILE = os.path.abspath('audio/test_audio.mp3')
if not os.path.exists(AUDIO_FILE):
    raise FileNotFoundError(f'Audio not found: {AUDIO_FILE}. Re-run Cell 4.')

dur = subprocess.run(
    ['ffprobe', '-v', 'error', '-show_entries', 'format=duration',
     '-of', 'default=noprint_wrappers=1:nokey=1', AUDIO_FILE],
    capture_output=True, text=True, check=True,
)
dur_min = float(dur.stdout.strip()) / 60
print(f'Using audio: {AUDIO_FILE} ({dur_min:.1f} min)')
if dur_min < 50:
    raise RuntimeError(f'Audio is only {dur_min:.1f} min — re-run Cell 4 to download 55-min clip.')

from main import run_pipeline
run_pipeline(AUDIO_FILE, start_stage=1)

FileNotFoundError: Audio not found: /content/urdu-pipeline/audio/test_audio.mp3. Re-run Cell 4.